In [15]:
import tensorflow as tf
print(tf.__version__)
print('Eager:', tf.executing_eagerly())

2.20.0
Eager: True


In [16]:
!pip install pandas scikit-learn matplotlib pillow requests tqdm

In [17]:
# -*- coding: utf-8 -*-
"""
Training pipeline: Label Studio (export JSON) -> Image Classification (Keras/TensorFlow), CPU friendly

- Parse export JSON Label Studio -> (image_path, label)
- Gère: /data/local-files/?d=..., URLs http(s), chemins relatifs/absolus
- Fix: suppression segments dupliqués (ex. ...\test_set\test_set\...), fallback de recherche par nom
- Filtrage strict anti-bruit: ignore fichiers cachés/système, extensions non image, fichiers vides
- Split stratifié train/val/test
- tf.data + data augmentation (légère)
- Transfer learning EfficientNetB0 (ImageNet) + fine-tuning partiel
- Callbacks (EarlyStopping, ModelCheckpoint)
- Évaluation + artefacts: best_model.keras, final_model.keras, labels.json, courbes, matrice de confusion
"""

import os
import json
import glob
import hashlib
import shutil
import random
import warnings
from typing import List, Tuple, Dict, Any
from urllib.parse import urlparse, parse_qs, unquote

import requests
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import matplotlib.pyplot as plt

# ======================
# ====== CONFIG =========
# ======================

# 1) Adapte CE chemin vers ton export JSON Label Studio :
LS_EXPORT_JSON = r"C:\Users\hp\PyCharmMiscProject\project-4-at-2025-10-04-08-38-2742532f.json"

# 2) Racine alternative pour retrouver un fichier par son nom si le chemin exact n'existe pas (OneDrive, etc.)
FALLBACK_SEARCH_ROOT = r"C:\Users\hp\Downloads\Chiens_Chats"

# Si les chemins dans le JSON sont relatifs, aide la résolution ici (sinon laisse None).
BASE_IMAGE_DIR = None

PRINT_DEBUG = True  # affiche des exemples de chemins résolus

WORKDIR = "runs_cls"
os.makedirs(WORKDIR, exist_ok=True)
URL_CACHE_DIR = os.path.join(WORKDIR, "url_cache")
os.makedirs(URL_CACHE_DIR, exist_ok=True)

# Hyperparamètres (CPU friendly)
IMG_SIZE = (160, 160)    # plus rapide que 224x224 en CPU
BATCH_SIZE = 32
EPOCHS = 12
VAL_SIZE = 0.15
TEST_SIZE = 0.15
SEED = 42

# (Optionnel) Stabiliser l'utilisation CPU (à ajuster selon ta machine)
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

# Extensions strictement autorisées (décommente ".webp" si supporté par ta build TensorFlow)
ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}  # , ".webp"

# ======================
# ===== UTILITAIRES =====
# ======================

def is_url(path: str) -> bool:
    return isinstance(path, str) and (path.startswith("http://") or path.startswith("https://"))

def safe_join(base_dir: str, maybe_rel: str) -> str:
    if base_dir is None:
        return os.path.normpath(maybe_rel)
    return os.path.normpath(os.path.join(base_dir, maybe_rel))

def is_hidden_or_system(path: str) -> bool:
    name = os.path.basename(path)
    return (name.startswith(".") or name.startswith("._") or
            name.lower() in {"thumbs.db", "desktop.ini", "_ds_store", ".ds_store"})

def has_allowed_ext(path: str) -> bool:
    return os.path.splitext(path)[1].lower() in ALLOWED_EXTS

def url_to_cache_path(url: str) -> str:
    h = hashlib.sha256(url.encode("utf-8")).hexdigest()[:20]
    ext = os.path.splitext(url.split("?")[0])[-1].lower()
    if ext not in ALLOWED_EXTS:
        ext = ".jpg"
    return os.path.join(URL_CACHE_DIR, f"{h}{ext}")

def download_if_needed(url: str) -> str:
    """Télécharge l'URL dans le cache si absent. Retourne le chemin local."""
    dst = url_to_cache_path(url)
    if not os.path.exists(dst):
        try:
            with requests.get(url, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(dst, "wb") as f:
                    shutil.copyfileobj(r.raw, f)
        except Exception as e:
            warnings.warn(f"Échec téléchargement: {url} ({e})")
            return None
    return dst

def extract_label_from_result(result_obj: Dict[str, Any]) -> str:
    """Couvre les cas courants: value.choices, value.labels, value.choice."""
    val = result_obj.get("value", {})
    if "choices" in val and isinstance(val["choices"], list) and len(val["choices"]) > 0:
        return str(val["choices"][0])
    if "labels" in val and isinstance(val["labels"], list) and len(val["labels"]) > 0:
        return str(val["labels"][0])
    if "choice" in val and isinstance(val["choice"], str):
        return val["choice"]
    return None

def collapse_adjacent_duplicates(path_str: str) -> str:
    """
    Si un segment est dupliqué consécutivement (ex: ...\\test_set\\test_set\\...),
    on le réduit en un seul (...\\test_set\\...).
    """
    norm = os.path.normpath(path_str)
    parts = norm.split(os.sep)
    collapsed = []
    for p in parts:
        if not collapsed or collapsed[-1].lower() != p.lower():
            collapsed.append(p)
    return os.sep.join(collapsed)

def smart_search_by_filename(filename: str, search_root: str) -> str:
    """
    Fallback: cherche filename sous search_root (1er match).
    Attention: peut être coûteux si l'arbre est énorme.
    """
    if not search_root or not os.path.isdir(search_root):
        return None
    pattern = os.path.join(search_root, "**", filename)
    matches = glob.glob(pattern, recursive=True)
    if matches:
        return os.path.normpath(matches[0])
    return None

def resolve_ls_local_file(uri: str) -> str:
    """
    '/data/local-files/?d=Users%5C...%5Ccats%5Ccat.123.jpg' -> 'C:\\Users\\...\\cats\\cat.123.jpg' (Windows)
    + réduction segments dupliqués + normalisation.
    """
    try:
        parsed = urlparse(uri)
        if parsed.path.startswith("/data/local-files/"):
            qs = parse_qs(parsed.query)
            if "d" in qs and len(qs["d"]) > 0:
                raw = qs["d"][0]              # 'Users%5CScriptLab%5C...'
                win_path = unquote(raw)       # 'Users\\ScriptLab\\...'
                # Si pas de lettre de lecteur → préfixer SystemDrive
                drive, _ = os.path.splitdrive(win_path)
                if drive == "":
                    system_drive = os.environ.get("SystemDrive", "C:")
                    if not win_path.startswith(("\\", "/")):
                        win_path = os.path.join(system_drive, win_path)  # C:\Users\...
                    else:
                        win_path = system_drive + win_path               # C:\\Users\...
                # Réduction doublons
                win_path = collapse_adjacent_duplicates(win_path)
                return os.path.normpath(win_path)
    except Exception as e:
        warnings.warn(f"Echec resolve_ls_local_file pour {uri}: {e}")
    return None

def resolve_image_path(image_path: str, base_image_dir: str = None) -> str:
    """
    Résout: (1) /data/local-files, (2) URL, (3) chemin local.
    Fallbacks:
      - réduction segments dupliqués,
      - recherche par nom de fichier sous FALLBACK_SEARCH_ROOT si le chemin n'existe pas.
    """
    candidate = None

    # 1) Label Studio local-files
    if isinstance(image_path, str) and image_path.startswith("/data/local-files/?"):
        candidate = resolve_ls_local_file(image_path)

    # 2) URL http(s)
    elif is_url(image_path):
        return download_if_needed(image_path)

    # 3) Chemin local (relatif/absolu)
    else:
        candidate = safe_join(base_image_dir, image_path) if base_image_dir else os.path.normpath(image_path)

    if candidate:
        # Essai direct
        if os.path.exists(candidate):
            return candidate
        # Essai après réduction des doublons
        c2 = collapse_adjacent_duplicates(candidate)
        if c2 != candidate and os.path.exists(c2):
            return c2
        # Essai recherche par nom de fichier (fallback)
        filename = os.path.basename(candidate)
        alt = smart_search_by_filename(filename, FALLBACK_SEARCH_ROOT)
        if alt and os.path.exists(alt):
            return alt

    return None

def parse_labelstudio_export(json_path: str, base_image_dir: str = None) -> pd.DataFrame:
    """
    Retourne DataFrame['image_path','label'] à partir d'un export Label Studio.
    Filtre strictement les images (extensions autorisées, non cachées, taille > 0).
    """
    with open(json_path, "r", encoding="utf-8") as f:
        tasks = json.load(f)

    rows, skipped_nonimg, skipped_missing, skipped_empty = [], 0, 0, 0

    for task in tasks:
        data = task.get("data", {})
        # champs possibles
        img_key_candidates = ["image", "img", "image_url", "imageUrl", "Image"]
        image_path = None
        for k in img_key_candidates:
            if k in data:
                image_path = data[k]
                break
        if image_path is None:
            # Recherche large si structure différente
            for v in data.values():
                if isinstance(v, str) and (is_url(v) or v.startswith("/data/local-files/")
                                           or v.lower().endswith(tuple(ALLOWED_EXTS))):
                    image_path = v
                    break
        if image_path is None:
            continue

        # Récup label
        anns = task.get("annotations", [])
        label = None
        for ann in anns:
            if ann.get("was_cancelled", False):
                continue
            for res in ann.get("result", []):
                maybe = extract_label_from_result(res)
                if maybe:
                    label = maybe
                    break
            if label:
                break
        if not label:
            continue

        # Résolution du chemin
        local_path = resolve_image_path(image_path, base_image_dir)
        if not local_path or not os.path.exists(local_path):
            skipped_missing += 1
            continue

        # Filtres anti-bruit
        if is_hidden_or_system(local_path) or not has_allowed_ext(local_path):
            skipped_nonimg += 1
            continue
        try:
            if os.path.getsize(local_path) <= 0:
                skipped_empty += 1
                continue
        except OSError:
            skipped_missing += 1
            continue

        rows.append({"image_path": os.path.normpath(local_path), "label": label})

    df = pd.DataFrame(rows)
    print(f"[FILTER] ignorés -> non-images: {skipped_nonimg}, manquants: {skipped_missing}, vides: {skipped_empty}")
    if df.empty:
        raise RuntimeError("Aucune (image, label) valide trouvée dans l'export Label Studio.")
    return df

# ======================
# ==== DATA PIPELINE ===
# ======================

def make_label_mapping(labels: List[str]) -> Tuple[Dict[str, int], Dict[int, str]]:
    classes = sorted(list(set(labels)))
    str2id = {c: i for i, c in enumerate(classes)}
    id2str = {i: c for c, i in str2id.items()}
    return str2id, id2str

def load_image_tf(path, img_size=IMG_SIZE):
    # Décodeur "strict" basé sur l'extension (évite les GIF animés, .DS_Store, etc.)
    img_bytes = tf.io.read_file(path)
    lower = tf.strings.lower(path)
    is_jpg = tf.strings.regex_full_match(lower, r".*\.(jpg|jpeg)$")
    is_png = tf.strings.regex_full_match(lower, r".*\.png$")
    is_bmp = tf.strings.regex_full_match(lower, r".*\.bmp$")
    def _decode_jpg(): return tf.image.decode_jpeg(img_bytes, channels=3)
    def _decode_png(): return tf.image.decode_png(img_bytes, channels=3)
    def _decode_bmp(): return tf.image.decode_bmp(img_bytes)
    img = tf.case(
        [(is_jpg, _decode_jpg), (is_png, _decode_png), (is_bmp, _decode_bmp)],
        default=lambda: tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    )
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def build_tf_dataset(paths: List[str], labels: List[int], batch_size: int, training: bool):
    ds_paths = tf.constant(paths)
    ds_labels = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((ds_paths, ds_labels))

    def _loader(p, y):
        img = load_image_tf(p)
        return img, y

    ds = ds.map(_loader, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        aug = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.05),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.1),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(buffer_size=min(len(paths), 1000), seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# ======================
# ====== MODELE =========
# ======================

def build_model(num_classes: int, input_shape=(160, 160, 3)) -> Model:
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=input_shape)
    for layer in base.layers:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=base.input, outputs=outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# ======================
# ====== MAIN RUN =======
# ======================

def main():
    print("==> Lecture export Label Studio ...")
    df = parse_labelstudio_export(LS_EXPORT_JSON, base_image_dir=BASE_IMAGE_DIR)
    print(df.head())
    print(f"Total images: {len(df)} | Classes: {sorted(df['label'].unique())}")

    # Debug chemins résolus
    if PRINT_DEBUG:
        print("\n[DEBUG] Exemples de chemins résolus :")
        for i, p in enumerate(df["image_path"].head(5).tolist()):
            print(f"  {i+1:02d}. {p} | exists={os.path.exists(p)}")
        missing = [p for p in df["image_path"] if not os.path.exists(p)]
        print(f"[DEBUG] Images introuvables après résolution: {len(missing)}")

    # Mapping labels
    str2id, id2str = make_label_mapping(df["label"].tolist())
    df["label_id"] = df["label"].map(str2id)

    # Split stratifié train/val/test
    df_trainval, df_test = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label_id"]
    )
    rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
    df_train, df_val = train_test_split(
        df_trainval, test_size=rel_val, random_state=SEED, stratify=df_trainval["label_id"]
    )

    print(f"Split -> train: {len(df_train)} | val: {len(df_val)} | test: {len(df_test)}")

    # Sécurité supplémentaire côté DataFrame (parano)
    def _keep_ok(p):
        try:
            return (os.path.exists(p) and os.path.getsize(p) > 0 and
                    has_allowed_ext(p) and not is_hidden_or_system(p))
        except Exception:
            return False

    for name, dframe in [("train", df_train), ("val", df_val), ("test", df_test)]:
        before = len(dframe)
        dframe.drop(index=[i for i, p in dframe["image_path"].items() if not _keep_ok(p)], inplace=True)
        dframe.reset_index(drop=True, inplace=True)
        after = len(dframe)
        if before != after:
            print(f"[CLEAN] {name}: retiré {before-after} entrées invalides (restant {after})")

    # Datasets tf.data
    train_ds = build_tf_dataset(df_train["image_path"].tolist(), df_train["label_id"].tolist(), BATCH_SIZE, training=True)
    val_ds   = build_tf_dataset(df_val["image_path"].tolist(),   df_val["label_id"].tolist(),   BATCH_SIZE, training=False)
    test_ds  = build_tf_dataset(df_test["image_path"].tolist(),  df_test["label_id"].tolist(),  BATCH_SIZE, training=False)

    # Poids de classes
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(df["label_id"]),
        y=df_train["label_id"].values
    )
    class_weights = {i: w for i, w in enumerate(class_weights)}
    print("Class weights:", class_weights)

    # Modèle
    model = build_model(num_classes=len(str2id), input_shape=(*IMG_SIZE, 3))
    model.summary()

    # Callbacks
    ckpt_path = os.path.join(WORKDIR, "best_model.keras")
    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=5, mode="max", restore_best_weights=True),
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", mode="max", save_best_only=True)
    ]

    # Phase 1: backbone gelé
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks
    )

    # Fine-tuning partiel (dé-geler une queue de couches, BN laissées gelées)
    unfreeze_from = max(0, len(model.layers) - 60)
    for layer in model.layers[unfreeze_from:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    history_ft = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=max(5, EPOCHS // 2),
        class_weight=class_weights,
        callbacks=callbacks
    )

    # Sauvegardes
    model.save(os.path.join(WORKDIR, "final_model.keras"))
    with open(os.path.join(WORKDIR, "labels.json"), "w", encoding="utf-8") as f:
        json.dump({"str2id": str2id, "id2str": id2str}, f, ensure_ascii=False, indent=2)

    # Courbes
    def plot_history(h, title, outpng):
        plt.figure(figsize=(6,4))
        plt.plot(h.history.get("accuracy", []), label="train_acc")
        plt.plot(h.history.get("val_accuracy", []), label="val_acc")
        plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(title)
        plt.legend(); plt.tight_layout(); plt.savefig(outpng, dpi=150); plt.close()

    plot_history(history,    "Training (frozen backbone)", os.path.join(WORKDIR, "history_frozen.png"))
    plot_history(history_ft, "Fine-tuning (unfrozen tail)", os.path.join(WORKDIR, "history_finetune.png"))

    # Évaluation
    print("\n==> Évaluation sur jeu de test")
    y_true = df_test["label_id"].tolist()
    y_pred = []
    for batch_imgs, _ in test_ds:
        probs = model.predict(batch_imgs, verbose=0)
        y_pred.extend(np.argmax(probs, axis=1))
    target_names = [id2str[i] for i in sorted(id2str)]
    print(classification_report(y_true, y_pred, target_names=target_names))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(id2str))))
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title("Confusion Matrix")
    ticks = np.arange(len(id2str))
    ax.set_xticks(ticks); ax.set_xticklabels([id2str[i] for i in ticks], rotation=45, ha="right")
    ax.set_yticks(ticks); ax.set_yticklabels([id2str[i] for i in ticks])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.ylabel("True label"); plt.xlabel("Predicted label")
    cm_path = os.path.join(WORKDIR, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=150); plt.close()

    print(f"\nArtefacts sauvegardés dans: {os.path.abspath(WORKDIR)}")
    print(f"- Meilleur modèle: {ckpt_path}")
    print(f"- Modèle final:   {os.path.join(WORKDIR, 'final_model.keras')}")
    print(f"- Labels map:     {os.path.join(WORKDIR, 'labels.json')}")
    print(f"- Courbes:        history_frozen.png, history_finetune.png")
    print(f"- Matrice conf:   {cm_path}")

if __name__ == "__main__":
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    tf.keras.utils.set_random_seed(SEED)
    main()


==> Lecture export Label Studio ...


<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
C:\Users\hp\AppData\Local\Temp\ipykernel_5588\3641743120.py:2: SyntaxWarning: invalid escape sequence '\.'
  """


[FILTER] ignorés -> non-images: 0, manquants: 51, vides: 0


RuntimeError: Aucune (image, label) valide trouvée dans l'export Label Studio.

In [1]:
model.fit(X_train, y_train, class_weight={0:1, 1:1})

NameError: name 'model' is not defined